# Gold — Volume mensal de entregas por estado com MoM

Desenvolvido por: Ygor Moraes

## Objetivo

Criar a Gold `gold_ecommerce_rastreamento_entregas_volume_estado_mom`, medindo o crescimento mês contra mês do volume de pedidos entregues por estado.

## Regra de negócio

Considerar apenas pedidos entregues a partir da Silver de rastreamento:

`status_entrega = "entregue"`

Para obter o estado da entrega, são usados os joins:

- `rastreamento.id_pedido_ecommerce -> pedidos.id_pedido`
- `pedidos.id_endereco_entrega -> enderecos.id_endereco`

A Gold calcula:

- quantidade de pedidos entregues por estado e mês;
- quantidade do mês anterior;
- variação absoluta MoM;
- crescimento percentual MoM.

## Fontes

- Silver `ecommerce_rastreamento_entregas`
- Silver `ecommerce_pedidos`
- Silver `ecommerce_enderecos`

## Cuidados técnicos

- `ecommerce_rastreamento_entregas` é lida como Delta.
- `ecommerce_pedidos` é lida como Delta.
- `ecommerce_enderecos` é lida como Delta.
- Pedidos devem ser deduplicados por `id_pedido`.
- Endereços devem ser deduplicados por `id_endereco`.
- Os joins devem ser validados para não inflar o volume de entregas.
- A Gold deve manter uma linha por estado, ano e mês.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa funções e define parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    countDistinct,
    current_timestamp,
    lag,
    month,
    round as spark_round,
    sum as spark_sum,
    to_date,
    to_timestamp,
    trim,
    upper,
    when,
    year,
    row_number
)

from pyspark.sql.window import Window

SILVER_RASTREAMENTO_TABLE = "ecommerce_rastreamento_entregas"
SILVER_PEDIDOS_TABLE = "ecommerce_pedidos"
SILVER_ENDERECOS_TABLE = "ecommerce_enderecos"

SILVER_RASTREAMENTO_PATH = f"{SILVER_BASE_PATH}{SILVER_RASTREAMENTO_TABLE}"
SILVER_PEDIDOS_PATH = f"{SILVER_BASE_PATH}{SILVER_PEDIDOS_TABLE}"
SILVER_ENDERECOS_PATH = f"{SILVER_BASE_PATH}{SILVER_ENDERECOS_TABLE}"

GOLD_TABLE = "gold_ecommerce_rastreamento_entregas_volume_estado_mom"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

STATUS_ENTREGA_CONSIDERADO = "entregue"

RASTREAMENTO_REQUIRED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "status_entrega",
    "dt_evento"
]

PEDIDOS_REQUIRED_COLUMNS = [
    "id_pedido",
    "id_endereco_entrega",
    "dt_pedido",
    "dt_ultima_atualizacao_status",
    "silver_processed_at"
]

ENDERECOS_REQUIRED_COLUMNS = [
    "id_endereco",
    "estado",
    "silver_processed_at"
]

GOLD_KEY_COLUMNS = [
    "estado",
    "ano_entrega",
    "mes_entrega"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_RASTREAMENTO_PATH:", SILVER_RASTREAMENTO_PATH)
print("SILVER_PEDIDOS_PATH:", SILVER_PEDIDOS_PATH)
print("SILVER_ENDERECOS_PATH:", SILVER_ENDERECOS_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers necessárias para montar a Gold.

df_rastreamento = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_RASTREAMENTO_PATH)
)

df_pedidos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PEDIDOS_PATH)
)

df_enderecos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_ENDERECOS_PATH)
)

total_rastreamento = df_rastreamento.count()
total_pedidos = df_pedidos.count()
total_enderecos = df_enderecos.count()

print("Silvers lidas com sucesso.")
print(f"Total eventos rastreamento: {total_rastreamento}")
print(f"Total pedidos: {total_pedidos}")
print(f"Total endereços: {total_enderecos}")

In [0]:
# Valida colunas obrigatórias, volumes e duplicidades esperadas nas fontes.

validate_required_columns(df_rastreamento, RASTREAMENTO_REQUIRED_COLUMNS)
validate_required_columns(df_pedidos, PEDIDOS_REQUIRED_COLUMNS)
validate_required_columns(df_enderecos, ENDERECOS_REQUIRED_COLUMNS)

rastreamento_distintos = (
    df_rastreamento
    .select(col("id_rastreamento").cast("int").alias("id_rastreamento"))
    .distinct()
    .count()
)

pedidos_distintos = (
    df_pedidos
    .select(col("id_pedido").cast("int").alias("id_pedido"))
    .distinct()
    .count()
)

enderecos_distintos = (
    df_enderecos
    .select(col("id_endereco").cast("int").alias("id_endereco"))
    .distinct()
    .count()
)

rastreamento_duplicados = total_rastreamento - rastreamento_distintos
pedidos_duplicados = total_pedidos - pedidos_distintos
enderecos_duplicados = total_enderecos - enderecos_distintos

eventos_sem_pedido = df_rastreamento.filter(col("id_pedido_ecommerce").isNull()).count()
eventos_sem_status = df_rastreamento.filter(col("status_entrega").isNull()).count()
eventos_sem_dt_evento = df_rastreamento.filter(col("dt_evento").isNull()).count()

pedidos_sem_id = df_pedidos.filter(col("id_pedido").isNull()).count()
pedidos_sem_endereco = df_pedidos.filter(col("id_endereco_entrega").isNull()).count()

enderecos_sem_id = df_enderecos.filter(col("id_endereco").isNull()).count()
enderecos_sem_estado = df_enderecos.filter(col("estado").isNull()).count()

print(f"Eventos rastreamento: {total_rastreamento}")
print(f"Eventos duplicados por id_rastreamento: {rastreamento_duplicados}")
print(f"Eventos sem id_pedido_ecommerce: {eventos_sem_pedido}")
print(f"Eventos sem status_entrega: {eventos_sem_status}")
print(f"Eventos sem dt_evento: {eventos_sem_dt_evento}")

print(f"Pedidos: {total_pedidos}")
print(f"Pedidos distintos por id_pedido: {pedidos_distintos}")
print(f"Pedidos duplicados por id_pedido: {pedidos_duplicados}")
print(f"Pedidos sem id_pedido: {pedidos_sem_id}")
print(f"Pedidos sem id_endereco_entrega: {pedidos_sem_endereco}")

print(f"Endereços: {total_enderecos}")
print(f"Endereços distintos por id_endereco: {enderecos_distintos}")
print(f"Endereços duplicados por id_endereco: {enderecos_duplicados}")
print(f"Endereços sem id_endereco: {enderecos_sem_id}")
print(f"Endereços sem estado: {enderecos_sem_estado}")

if total_rastreamento == 0:
    raise Exception("Erro: a Silver de rastreamento está vazia.")

if total_pedidos == 0:
    raise Exception("Erro: a Silver de pedidos está vazia.")

if total_enderecos == 0:
    raise Exception("Erro: a Silver de endereços está vazia.")

if rastreamento_duplicados > 0:
    raise Exception("Erro: existem eventos duplicados por id_rastreamento.")

if eventos_sem_pedido > 0:
    raise Exception("Erro: existem eventos sem id_pedido_ecommerce.")

if eventos_sem_status > 0:
    raise Exception("Erro: existem eventos sem status_entrega.")

if eventos_sem_dt_evento > 0:
    raise Exception("Erro: existem eventos sem dt_evento.")

if pedidos_sem_id > 0:
    raise Exception("Erro: existem pedidos sem id_pedido.")

if pedidos_sem_endereco > 0:
    raise Exception("Erro: existem pedidos sem id_endereco_entrega.")

if enderecos_sem_id > 0:
    raise Exception("Erro: existem endereços sem id_endereco.")

if enderecos_sem_estado > 0:
    raise Exception("Erro: existem endereços sem estado.")

print("Validação OK: fontes mínimas conferidas.")

In [0]:
# Deduplica pedidos mantendo um registro por id_pedido.

window_pedidos = (
    Window
    .partitionBy("id_pedido")
    .orderBy(
        col("silver_processed_at").desc_nulls_last(),
        col("dt_ultima_atualizacao_status").desc_nulls_last(),
        col("dt_pedido").desc_nulls_last()
    )
)

df_pedidos_dedup = (
    df_pedidos
    .withColumn("id_pedido", col("id_pedido").cast("int"))
    .withColumn("id_endereco_entrega", col("id_endereco_entrega").cast("int"))
    .withColumn("rn", row_number().over(window_pedidos))
    .filter(col("rn") == 1)
    .drop("rn")
)

total_pedidos_dedup = df_pedidos_dedup.count()

pedidos_dedup_distintos = (
    df_pedidos_dedup
    .select("id_pedido")
    .distinct()
    .count()
)

pedidos_duplicados_apos_dedup = total_pedidos_dedup - pedidos_dedup_distintos

print(f"Total pedidos original: {total_pedidos}")
print(f"Total pedidos após deduplicação: {total_pedidos_dedup}")
print(f"Pedidos duplicados após deduplicação: {pedidos_duplicados_apos_dedup}")

if pedidos_duplicados_apos_dedup > 0:
    raise Exception("Erro: ainda existem pedidos duplicados após deduplicação.")

print("Validação OK: pedidos deduplicados por id_pedido.")

In [0]:
# Deduplica endereços mantendo um registro por id_endereco.

window_enderecos = (
    Window
    .partitionBy("id_endereco")
    .orderBy(col("silver_processed_at").desc_nulls_last())
)

df_enderecos_dedup = (
    df_enderecos
    .withColumn("id_endereco", col("id_endereco").cast("int"))
    .withColumn("estado", upper(trim(col("estado"))))
    .withColumn("rn", row_number().over(window_enderecos))
    .filter(col("rn") == 1)
    .drop("rn")
)

total_enderecos_dedup = df_enderecos_dedup.count()

enderecos_dedup_distintos = (
    df_enderecos_dedup
    .select("id_endereco")
    .distinct()
    .count()
)

enderecos_duplicados_apos_dedup = total_enderecos_dedup - enderecos_dedup_distintos

print(f"Total endereços original: {total_enderecos}")
print(f"Total endereços após deduplicação: {total_enderecos_dedup}")
print(f"Endereços duplicados após deduplicação: {enderecos_duplicados_apos_dedup}")

if enderecos_duplicados_apos_dedup > 0:
    raise Exception("Erro: ainda existem endereços duplicados após deduplicação.")

print("Validação OK: endereços deduplicados por id_endereco.")

In [0]:
# Filtra entregas concluídas e mantém uma entrega por pedido.

df_entregas = (
    df_rastreamento
    .filter(col("status_entrega") == STATUS_ENTREGA_CONSIDERADO)
    .select(
        col("id_rastreamento").cast("int").alias("id_rastreamento"),
        col("id_pedido_ecommerce").cast("int").alias("id_pedido_ecommerce"),
        col("dt_evento")
    )
    .withColumn("ano_entrega", year(col("dt_evento")))
    .withColumn("mes_entrega", month(col("dt_evento")))
)

window_entregas = (
    Window
    .partitionBy("id_pedido_ecommerce")
    .orderBy(col("dt_evento").desc_nulls_last())
)

df_entregas_dedup = (
    df_entregas
    .withColumn("rn", row_number().over(window_entregas))
    .filter(col("rn") == 1)
    .drop("rn")
)

total_entregas = df_entregas.count()
total_entregas_dedup = df_entregas_dedup.count()

pedidos_entregues_distintos = (
    df_entregas_dedup
    .select("id_pedido_ecommerce")
    .distinct()
    .count()
)

pedidos_entregues_duplicados = total_entregas_dedup - pedidos_entregues_distintos

print(f"Total eventos com status entregue: {total_entregas}")
print(f"Total entregas após deduplicação por pedido: {total_entregas_dedup}")
print(f"Pedidos entregues distintos: {pedidos_entregues_distintos}")
print(f"Pedidos entregues duplicados após deduplicação: {pedidos_entregues_duplicados}")

if total_entregas_dedup == 0:
    raise Exception("Erro: nenhum pedido entregue encontrado.")

if pedidos_entregues_duplicados > 0:
    raise Exception("Erro: ainda existem pedidos entregues duplicados após deduplicação.")

print("Validação OK: entregas deduplicadas por pedido.")

In [0]:
# Cruza entregas com pedidos e endereços para obter o estado da entrega.

df_entregas_estado = (
    df_entregas_dedup
    .join(
        df_pedidos_dedup.select(
            col("id_pedido").alias("id_pedido_ecommerce"),
            "id_endereco_entrega"
        ),
        on="id_pedido_ecommerce",
        how="left"
    )
    .join(
        df_enderecos_dedup.select(
            col("id_endereco").alias("id_endereco_entrega"),
            "estado"
        ),
        on="id_endereco_entrega",
        how="left"
    )
)

total_entregas_estado = df_entregas_estado.count()

pedidos_entregas_estado_distintos = (
    df_entregas_estado
    .select("id_pedido_ecommerce")
    .distinct()
    .count()
)

pedidos_duplicados_join = total_entregas_estado - pedidos_entregas_estado_distintos

entregas_sem_endereco = (
    df_entregas_estado
    .filter(col("id_endereco_entrega").isNull())
    .count()
)

entregas_sem_estado = (
    df_entregas_estado
    .filter(col("estado").isNull())
    .count()
)

print(f"Total entregas deduplicadas: {total_entregas_dedup}")
print(f"Total após joins: {total_entregas_estado}")
print(f"Pedidos distintos após joins: {pedidos_entregas_estado_distintos}")
print(f"Pedidos duplicados após joins: {pedidos_duplicados_join}")
print(f"Entregas sem endereço: {entregas_sem_endereco}")
print(f"Entregas sem estado: {entregas_sem_estado}")

if total_entregas_estado != total_entregas_dedup:
    raise Exception("Erro: joins alteraram o total de entregas.")

if pedidos_duplicados_join > 0:
    raise Exception("Erro: joins geraram duplicidade de pedidos.")

if entregas_sem_endereco > 0:
    raise Exception("Erro: existem entregas sem endereço após join com pedidos.")

if entregas_sem_estado > 0:
    raise Exception("Erro: existem entregas sem estado após join com endereços.")

print("Validação OK: entregas enriquecidas com estado sem inflar volume.")

In [0]:
# Agrupa entregas por estado, ano e mês.

df_volume_estado_mensal = (
    df_entregas_estado
    .groupBy(
        "estado",
        "ano_entrega",
        "mes_entrega"
    )
    .agg(
        countDistinct("id_pedido_ecommerce").alias("qtd_pedidos_entregues")
    )
)

total_volume_estado_mensal = df_volume_estado_mensal.count()

print(f"Total linhas da base mensal por estado: {total_volume_estado_mensal}")
display(df_volume_estado_mensal.orderBy("estado", "ano_entrega", "mes_entrega"))

In [0]:
# Calcula variação mês contra mês por estado.

window_estado_mom = (
    Window
    .partitionBy("estado")
    .orderBy("ano_entrega", "mes_entrega")
)

df_gold = (
    df_volume_estado_mensal
    .withColumn(
        "qtd_pedidos_entregues_mes_anterior",
        lag("qtd_pedidos_entregues").over(window_estado_mom)
    )
    .withColumn(
        "qtd_pedidos_entregues_mes_anterior",
        when(col("qtd_pedidos_entregues_mes_anterior").isNull(), 0)
        .otherwise(col("qtd_pedidos_entregues_mes_anterior"))
    )
    .withColumn(
        "variacao_absoluta_mom",
        col("qtd_pedidos_entregues") - col("qtd_pedidos_entregues_mes_anterior")
    )
    .withColumn(
        "crescimento_mom_percentual",
        when(
            col("qtd_pedidos_entregues_mes_anterior") == 0,
            None
        ).otherwise(
            spark_round(
                (
                    col("variacao_absoluta_mom") /
                    col("qtd_pedidos_entregues_mes_anterior")
                ) * 100,
                2
            )
        )
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("estado", "ano_entrega", "mes_entrega")
)

print("Gold de volume mensal por estado com MoM criada em memória.")
display(df_gold)

In [0]:
# Valida totais, chaves e campos principais da Gold.

total_linhas_gold = df_gold.count()

total_chaves_gold = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_gold = total_linhas_gold - total_chaves_gold

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues")
    )
    .collect()[0]
)

nulos_principais_gold = (
    df_gold
    .filter(
        col("estado").isNull() |
        col("ano_entrega").isNull() |
        col("mes_entrega").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("qtd_pedidos_entregues_mes_anterior").isNull() |
        col("variacao_absoluta_mom").isNull()
    )
    .count()
)

print(f"Total entregas deduplicadas: {total_entregas_dedup}")
print(f"Total pedidos entregues na Gold: {validacao_gold['total_pedidos_entregues']}")
print(f"Total linhas Gold: {total_linhas_gold}")
print(f"Chaves duplicadas Gold: {chaves_duplicadas_gold}")
print(f"Linhas com nulos principais: {nulos_principais_gold}")

if validacao_gold["total_pedidos_entregues"] != total_entregas_dedup:
    raise Exception("Erro: total de pedidos entregues da Gold não fecha com a base.")

if chaves_duplicadas_gold > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold.")

if nulos_principais_gold > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold.")

print("Validação OK: Gold em memória conferida.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .partitionBy("ano_entrega", "mes_entrega")
    .save(GOLD_PATH)
)

print(f"Gold gravada com sucesso em Delta: {GOLD_PATH}")

In [0]:
# Lê e valida a Gold Delta gravada.

df_gold_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

total_linhas_gold_saved = df_gold_saved.count()

total_chaves_gold_saved = (
    df_gold_saved
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_gold_saved = total_linhas_gold_saved - total_chaves_gold_saved

validacao_gold_saved = (
    df_gold_saved
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues")
    )
    .collect()[0]
)

nulos_principais_gold_saved = (
    df_gold_saved
    .filter(
        col("estado").isNull() |
        col("ano_entrega").isNull() |
        col("mes_entrega").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("qtd_pedidos_entregues_mes_anterior").isNull() |
        col("variacao_absoluta_mom").isNull()
    )
    .count()
)

print(f"Total linhas Gold Delta: {total_linhas_gold_saved}")
print(f"Chaves duplicadas Gold Delta: {chaves_duplicadas_gold_saved}")
print(f"Total entregas deduplicadas: {total_entregas_dedup}")
print(f"Total pedidos entregues Gold Delta: {validacao_gold_saved['total_pedidos_entregues']}")
print(f"Linhas com nulos principais Gold Delta: {nulos_principais_gold_saved}")

if validacao_gold_saved["total_pedidos_entregues"] != total_entregas_dedup:
    raise Exception("Erro: total de pedidos entregues da Gold Delta não confere.")

if chaves_duplicadas_gold_saved > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold Delta.")

if nulos_principais_gold_saved > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold Delta.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_saved
    .select(
        col("estado").cast("string").alias("estado"),
        col("ano_entrega").cast("int").alias("ano_entrega"),
        col("mes_entrega").cast("int").alias("mes_entrega"),
        col("qtd_pedidos_entregues").cast("int").alias("qtd_pedidos_entregues"),
        col("qtd_pedidos_entregues_mes_anterior").cast("int").alias("qtd_pedidos_entregues_mes_anterior"),
        col("variacao_absoluta_mom").cast("int").alias("variacao_absoluta_mom"),
        col("crescimento_mom_percentual").cast("decimal(10,2)").alias("crescimento_mom_percentual"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")
df_gold_sql.printSchema()
display(df_gold_sql.orderBy("estado", "ano_entrega", "mes_entrega"))

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final do SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_final = df_final.count()

total_chaves_final = (
    df_final
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_final = total_linhas_final - total_chaves_final

validacao_final = (
    df_final
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues")
    )
    .collect()[0]
)

nulos_principais_final = (
    df_final
    .filter(
        col("estado").isNull() |
        col("ano_entrega").isNull() |
        col("mes_entrega").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("qtd_pedidos_entregues_mes_anterior").isNull() |
        col("variacao_absoluta_mom").isNull()
    )
    .count()
)

print(f"Total linhas tabela final: {total_linhas_final}")
print(f"Chaves duplicadas tabela final: {chaves_duplicadas_final}")
print(f"Total entregas deduplicadas: {total_entregas_dedup}")
print(f"Total pedidos entregues tabela final: {validacao_final['total_pedidos_entregues']}")
print(f"Linhas com nulos principais tabela final: {nulos_principais_final}")

if validacao_final["total_pedidos_entregues"] != total_entregas_dedup:
    raise Exception("Erro: total de pedidos entregues da tabela final não confere.")

if chaves_duplicadas_final > 0:
    raise Exception("Erro: existem chaves duplicadas na tabela final.")

if nulos_principais_final > 0:
    raise Exception("Erro: existem nulos nas colunas principais da tabela final.")

print("Validação OK: tabela final SQL Server gravada corretamente.")